# Feature Engineering and Preprocessing

Cleans the data, handles missing values, fixes skewed features, engineers new features, and encodes everything for modeling. The actual logic lives in `src/preprocessing.py`, this notebook just calls it and shows what each step does.

In [1]:
import pandas as pd
import numpy as np
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append('../src')
from preprocessing import (
    remove_outliers,
    handle_missing_values,
    fix_skewness,
    engineer_features,
    encode_features
)

Loading the raw data.

In [2]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
train.shape, test.shape

((1460, 81), (1459, 80))

Separating the target and the id columns. SalePrice is log transformed since it is right skewed.

In [3]:
y = np.log1p(train['SalePrice'])
train_ID = train['Id']
test_ID = test['Id']
train.drop(['SalePrice', 'Id'], axis=1, inplace=True)
test.drop(['Id'], axis=1, inplace=True)

Removing outlier houses, then combining train and test so both get the exact same cleaning and encoding.

In [4]:
train, y = remove_outliers(train, y)

ntrain = train.shape[0]
ntest = test.shape[0]
all_data = pd.concat([train, test], axis=0, ignore_index=True)
all_data.shape

Removed 4 outliers. Train shape: (1456, 79)


(2915, 79)

Handling missing values.

In [5]:
all_data = handle_missing_values(all_data)

Missing values remaining: 0


Fixing skewed numerical features with a log1p transform.

In [6]:
all_data = fix_skewness(all_data)

Applied log1p to 20 skewed features


Engineering new features, total square footage, total bathrooms, house age, porch area, and a few binary flags and interaction terms.

In [7]:
all_data = engineer_features(all_data)

new_features = ['TotalSF', 'TotalBathrooms', 'HouseAge',
                'YearsSinceRemodel', 'TotalPorchSF', 'OverallScore']
all_data[new_features].describe()

Feature engineering complete. Shape: (2915, 90)


,TotalSF,TotalBathrooms,HouseAge,YearsSinceRemodel,TotalPorchSF,OverallScore
count,2915.000000,2915.000000,2915.000000,2915.000000,2915.000000,2915.00000
mean,1057.853281,2.205871,36.521784,23.553002,3.605946,33.71012
std,427.282950,0.806065,30.335130,20.894517,2.897307,9.16474
min,5.814131,1.000000,-1.000000,-2.000000,0.000000,1.00000
25%,802.469086,1.500000,7.000000,4.000000,0.000000,30.00000
50%,996.898715,2.000000,35.000000,15.000000,3.931826,35.00000
75%,1309.172425,2.500000,55.000000,43.000000,4.962845,40.00000
max,5103.536211,7.000000,136.000000,60.000000,16.071903,90.00000


Encoding categorical features. Quality columns use an ordinal map since order matters, other ordinal columns use label encoding, and the rest are one-hot encoded.

In [8]:
all_data = encode_features(all_data)

Encoding complete. Final shape: (2915, 238)


Splitting back into train and test.

In [9]:
X_train = all_data[:ntrain]
X_test = all_data[ntrain:]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y shape: {y.shape}")

X_train shape: (1456, 238)
X_test shape: (1459, 238)
y shape: (1456,)


In [10]:
assert X_train.isnull().sum().sum() == 0, "Missing values in train"
assert X_test.isnull().sum().sum() == 0, "Missing values in test"
print("No missing values")

No missing values


Saving the processed data for the modeling notebooks.

In [11]:
X_train.to_csv('../data/X_train_processed.csv', index=False)
X_test.to_csv('../data/X_test_processed.csv', index=False)
y.to_csv('../data/y_train.csv', index=False)